# Signal Filtering Impact on Scaffold Fingerprinting

Compare scaffold-based fingerprinting across different preprocessing pipelines:

**HCP 100 Unrelated** (Schaefer 300, LR):
- `no_hp`: No high-pass filter (essentially unfiltered parcellated signal)
- `hp`: HCP default high-pass filter (~2000s / 0.0005 Hz cutoff)

**HCP 1200 Subjects** (~1005, Schaefer 300, LR, NoGSR):
- `NoFilt`: No filtering
- `LowPassOnly`: Low-pass filter only
- `HighPassOnly`: High-pass filter only  
- `Bandpass`: Both high-pass and low-pass filters

Addresses reviewer concern: *"Why is topological structure so sensitive to high-frequency information?"*

In [ ]:
import sys
import sys
sys.path.append("..")
from topofinger import config

import os
import numpy as np
import pickle as pk
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import OrderedDict
from tqdm.notebook import tqdm
from scipy import stats
from pingouin import compute_effsize

from topofinger.hcp_IO import load_scaffold_schaefer300, save_correlations_subjects_fast
from topofinger.fingerprinting import compute_idiff, compute_effect_size, compute_success_rate
from topofinger.viz import plot_identifiability

%load_ext autoreload
%autoreload 2

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12

## Part 0: Compute Scaffold Similarity Matrices
Run this section after FC and scaffolds have been computed.
Computes `scaffold_correlations_whole.npy` for any variant that doesn't have it yet.

In [ ]:
# --- Check similarity matrices ---

# 1200 subjects variants: cortical-only (300 regions) matrices are built by scripts/06_similarity_matrices.py
base_1200 = config.HCP_1200 + ""
variants_1200 = ["NoGSR_LowPassOnly", "NoGSR_HighPassOnly", "NoGSR_NoFilt", "NoGSR_Bandpass"]

for variant in variants_1200:
    sim_path = os.path.join(base_1200, variant, "scaffolds", "whole", "scaffold_correlations_whole_cortex.npy")
    print(f"[{'OK' if os.path.exists(sim_path) else 'MISSING, run scripts/06_similarity_matrices.py'}] {variant}")

# 100 unrelated variants
base_100 = config.HCP_100 + "/schaefer300"
variants_100 = ["no_hp", "hp"]

for variant in variants_100:
    scaff_dir = os.path.join(base_100, variant, "scaffolds", "whole")
    freq_dir = os.path.join(scaff_dir, "freq")
    sim_path = os.path.join(scaff_dir, "scaffold_correlations_whole.npy")
    
    if os.path.exists(sim_path):
        print(f"[SKIP] 100unrelated/{variant}: similarity matrix already exists")
        continue
    
    print(f"[COMPUTING] 100unrelated/{variant} scaffold similarity matrix...")
    sc = load_scaffold_schaefer300(freq_dir + "/")
    print(f"  Loaded {len(sc)} subjects")
    save_correlations_subjects_fast(sc, sim_path.replace('.npy', ''))
    print(f"  Saved {sim_path}")

print("\nDone!")

In [ ]:
# FC similarity matrices for the 1200-subject variants (cortical-only, 300 regions)
# are built by scripts/06_similarity_matrices.py; here we only check they exist.
for variant in variants_1200:
    sim_path = os.path.join(base_1200, variant, "fc", "whole", "fc_correlations_whole_th1_cortex.npy")
    print(f"[{'OK' if os.path.exists(sim_path) else 'MISSING, run scripts/06_similarity_matrices.py'}] {variant} FC th1")


## Part 1: Load All Precomputed Similarity Matrices

In [ ]:
# --- Load all scaffold and FC similarity matrices ---

results = {}

# HCP 100 Unrelated
for variant, label in [("no_hp", "100unrel: No Filter"), ("hp", "100unrel: High-Pass")]:
    base = f"{config.HCP_100}/schaefer300/{variant}/scaffolds/whole"
    sim = np.load(os.path.join(base, "scaffold_correlations_whole.npy"))
    results[label] = {"scaffold_sim": sim, "n_subjects": sim.shape[0]}
    print(f"{label}: {sim.shape[0]} subjects")

# HCP 1200 Subjects (NoGSR variants)
label_map = {
    "NoGSR_NoFilt": "1200subj: No Filter",
    "NoGSR_LowPassOnly": "1200subj: Low-Pass Only",
    "NoGSR_HighPassOnly": "1200subj: High-Pass Only",
    "NoGSR_Bandpass": "1200subj: Bandpass",
}

for variant, label in label_map.items():
    base = f"{config.HCP_1200}/{variant}/scaffolds/whole"
    sim_path = os.path.join(base, "scaffold_correlations_whole_cortex.npy")
    if os.path.exists(sim_path):
        sim = np.load(sim_path)
        results[label] = {"scaffold_sim": sim, "n_subjects": sim.shape[0]}
        print(f"{label}: {sim.shape[0]} subjects")
    else:
        print(f"{label}: NOT AVAILABLE (run scaffold computation first)")

print(f"\nLoaded {len(results)} conditions")

In [ ]:
# --- Compute fingerprinting metrics for each condition ---

metrics = {}
for label, data in results.items():
    sim = data["scaffold_sim"]
    n = data["n_subjects"]
    
    idiff, i_self, i_others = compute_idiff(sim)
    d = compute_effect_size(sim)
    sr1, sr2 = compute_success_rate(sim)
    sr_mean = (sr1 + sr2) / 2
    
    metrics[label] = {
        "n_subjects": n,
        "idiff": idiff,
        "i_self": i_self,
        "i_others": i_others,
        "cohen_d": d,
        "sr_mean": sr_mean,
        "sr1": sr1,
        "sr2": sr2,
    }

# Print summary table
print(f"{'Condition':<30} {'N':>5} {'Cohen d':>8} {'SR (%)':>8} {'Idiff':>8} {'I_self':>8} {'I_others':>8}")
print("-" * 90)
for label, m in metrics.items():
    print(f"{label:<30} {m['n_subjects']:>5} {m['cohen_d']:>8.2f} {m['sr_mean']:>8.1f} {m['idiff']:>8.4f} {m['i_self']:>8.4f} {m['i_others']:>8.4f}")

## Part 2: Visualization

In [ ]:
# --- Similarity matrix heatmaps ---

conditions_100 = [k for k in results if k.startswith("100unrel")]
conditions_1200 = [k for k in results if k.startswith("1200subj")]

# Plot 100 unrelated
if len(conditions_100) > 0:
    fig, axes = plt.subplots(1, len(conditions_100), figsize=(6*len(conditions_100), 5))
    if len(conditions_100) == 1:
        axes = [axes]
    for ax, label in zip(axes, conditions_100):
        sim = results[label]["scaffold_sim"]
        plot_identifiability(sim, ax=ax, title=label.replace("100unrel: ", ""), kind='corr')
    fig.suptitle("HCP 100 Unrelated — Scaffold Fingerprinting", fontsize=16, y=1.05)
    plt.tight_layout()
    plt.savefig(config.FIGURES_RAW + "/filtering_100unrel_sim_matrices.png", dpi=200, bbox_inches='tight')
    plt.show()

# Plot 1200 subjects
if len(conditions_1200) > 0:
    fig, axes = plt.subplots(1, len(conditions_1200), figsize=(6*len(conditions_1200), 5))
    if len(conditions_1200) == 1:
        axes = [axes]
    for ax, label in zip(axes, conditions_1200):
        sim = results[label]["scaffold_sim"]
        plot_identifiability(sim, ax=ax, title=label.replace("1200subj: ", ""), kind='corr')
    fig.suptitle("HCP 1005 Subjects — Scaffold Fingerprinting (NoGSR)", fontsize=16, y=1.05)
    plt.tight_layout()
    plt.savefig(config.FIGURES_RAW + "/filtering_1200subj_sim_matrices.png", dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# --- Bar chart (Fig S5): scaffold vs FC (1%) across filtering conditions, HCP 1005 subjects ---

order = ["NoGSR_LowPassOnly", "NoGSR_Bandpass", "NoGSR_HighPassOnly", "NoGSR_NoFilt"]
names = [label_map[v].split(": ")[1] for v in order]
sc_d, sc_sr, fc_d, fc_sr = [], [], [], []
for v in order:
    m = metrics[label_map[v]]
    sc_d.append(m["cohen_d"]); sc_sr.append(m["sr_mean"])
    fc_sim = np.load(f"{config.HCP_1200}/{v}/fc/whole/fc_correlations_whole_th1_cortex.npy")
    fc_d.append(compute_effect_size(fc_sim))
    s1, s2 = compute_success_rate(fc_sim); fc_sr.append((s1 + s2) / 2)

colors = {"FC (1%)": "cornflowerblue", "Scaffolds": "sandybrown"}
x = np.arange(len(order)); w = 0.38
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (fc_vals, sc_vals, ylabel, title, fmt, off) in zip(axes, [
        (fc_d, sc_d, "Cohen's d", "Effect Size (Cohen's d)", "{:.2f}", 0.05),
        (fc_sr, sc_sr, "Success Rate (%)", "Identification Success Rate", "{:.1f}%", 1)]):
    for vals, shift, lab in [(fc_vals, -w / 2, "FC (1%)"), (sc_vals, w / 2, "Scaffolds")]:
        bars = ax.bar(x + shift, vals, w, color=colors[lab], edgecolor='black', linewidth=0.5, label=lab)
        for b, val in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + off, fmt.format(val), ha='center', va='bottom', fontsize=10)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha='right')
    ax.set_ylabel(ylabel); ax.set_title(title)
axes[0].set_ylim(0, max(sc_d) * 1.22)
axes[1].set_ylim(0, 112)
for ax in axes:
    ax.legend(loc='upper center', ncol=2)
plt.tight_layout()
plt.savefig(config.FIGURES_RAW + "/filtering_barplot_comparison.png", dpi=200, bbox_inches='tight')
plt.show()


## Part 3: Statistical Comparison (1200 Subjects)

Within the 1200-subject dataset, compare identifiability distributions across 4 filtering conditions using bootstrap confidence intervals on Cohen's d.

In [ ]:
# --- Bootstrap confidence intervals on Cohen's d for 1200 subjects ---

def bootstrap_cohen_d(sim_matrix, n_boot=1000, seed=42):
    """Bootstrap Cohen's d by resampling subjects with replacement.
    Pairs of a subject with its own copy are excluded from the inter-subject distribution."""
    rng = np.random.RandomState(seed)
    n = sim_matrix.shape[0]
    ds = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        sub_sim = sim_matrix[np.ix_(idx, idx)]
        inside = np.diag(sub_sim)
        outside = sub_sim[idx[:, None] != idx[None, :]]
        d = compute_effsize(inside, outside)
        ds.append(d)
    return np.array(ds)

# Compute for 1200 subject conditions
boot_results = {}
for label in conditions_1200:
    print(f"Bootstrapping {label}...")
    sim = results[label]["scaffold_sim"]
    ds = bootstrap_cohen_d(sim, n_boot=1000)
    boot_results[label] = ds
    ci_lo, ci_hi = np.percentile(ds, [2.5, 97.5])
    print(f"  Cohen's d = {np.mean(ds):.2f} [{ci_lo:.2f}, {ci_hi:.2f}]")

print()

In [ ]:
# --- Violin plot of bootstrapped Cohen's d distributions ---

if len(boot_results) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    labels_1200 = [l.replace("1200subj: ", "") for l in conditions_1200]
    data_1200 = [boot_results[l] for l in conditions_1200 if l in boot_results]
    
    parts = ax.violinplot(data_1200, positions=range(len(data_1200)), showmeans=True, showmedians=False)
    
    for pc in parts['bodies']:
        pc.set_facecolor('#FF9800')
        pc.set_alpha(0.7)
    
    ax.set_xticks(range(len(labels_1200)))
    ax.set_xticklabels(labels_1200, fontsize=12)
    ax.set_ylabel("Cohen's d (bootstrapped)", fontsize=13)
    ax.set_title("Effect of Filtering on Scaffold Fingerprinting — HCP 1005 Subjects", fontsize=14)
    
    # Add CI text
    for i, label in enumerate(conditions_1200):
        if label in boot_results:
            ds = boot_results[label]
            ci_lo, ci_hi = np.percentile(ds, [2.5, 97.5])
            ax.text(i, np.max(ds) + 0.05, f'd={metrics[label]["cohen_d"]:.2f}\n[{ci_lo:.2f}, {ci_hi:.2f}]',
                    ha='center', fontsize=9)
    
    ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label="Large effect (d=0.8)")
    ax.legend()
    ax.set_ylim(top=max(np.max(d) for d in data_1200) + 0.45)
    plt.tight_layout()
    plt.savefig(config.FIGURES_RAW + "/filtering_bootstrap_violin.png", dpi=200, bbox_inches='tight')
    plt.show()

## Part 4: Intra- vs Inter-Subject Similarity Distributions

Visualize the separation between diagonal (same subject across sessions) and off-diagonal (different subjects) correlations for each filtering condition.

In [ ]:
# --- Intra vs inter similarity distributions ---

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

all_conditions = list(results.keys())
for idx, label in enumerate(all_conditions):
    if idx >= 6:
        break
    ax = axes[idx]
    sim = results[label]["scaffold_sim"]
    n = sim.shape[0]
    
    intra = np.diag(sim)
    mask = ~np.eye(n, dtype=bool)
    inter = sim[mask]
    
    ax.hist(inter, bins=50, alpha=0.6, color='steelblue', density=True, label='Inter-subject')
    ax.hist(intra, bins=30, alpha=0.6, color='coral', density=True, label='Intra-subject')
    ax.axvline(np.mean(intra), color='red', linestyle='--', linewidth=1.5)
    ax.axvline(np.mean(inter), color='blue', linestyle='--', linewidth=1.5)
    ax.set_title(label.replace("1200subj", "1005 subjects").replace("100unrel", "100 unrelated"), fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlabel("Pearson correlation")

# Hide unused axes
for idx in range(len(all_conditions), 6):
    axes[idx].set_visible(False)

plt.suptitle("Intra- vs Inter-Subject Scaffold Similarity Distributions", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(config.FIGURES_RAW + "/filtering_intra_inter_distributions.png", dpi=200, bbox_inches='tight')
plt.show()

## Part 5: FC Fingerprinting Comparison

Compare scaffold vs FC fingerprinting under different filtering conditions (1200 subjects).

In [ ]:
# --- Load FC similarity matrices and compare with scaffold ---

fc_results = {}
for variant, label in label_map.items():
    fc_sim_path = f"{config.HCP_1200}/{variant}/fc/whole/fc_correlations_whole_th1_cortex.npy"
    if os.path.exists(fc_sim_path):
        sim = np.load(fc_sim_path)
        fc_results[label] = sim
        d = compute_effect_size(sim)
        sr1, sr2 = compute_success_rate(sim)
        print(f"FC {label}: d={d:.2f}, SR={(sr1+sr2)/2:.1f}%")
    else:
        print(f"FC {label}: NOT AVAILABLE")

# Compare scaffold vs FC
if len(fc_results) > 0:
    print("\n" + "="*80)
    print(f"{'Condition':<30} {'Scaffold d':>12} {'FC d':>12} {'Scaffold SR':>12} {'FC SR':>12}")
    print("-" * 80)
    for label in label_map.values():
        if label in metrics and label in fc_results:
            m = metrics[label]
            fc_sim = fc_results[label]
            fc_d = compute_effect_size(fc_sim)
            fc_sr1, fc_sr2 = compute_success_rate(fc_sim)
            fc_sr = (fc_sr1 + fc_sr2) / 2
            print(f"{label:<30} {m['cohen_d']:>12.2f} {fc_d:>12.2f} {m['sr_mean']:>12.1f} {fc_sr:>12.1f}")

## Part 6: Summary

Key findings about the impact of signal filtering on scaffold-based fingerprinting.

In [ ]:
# --- Summary statistics ---

print("=" * 80)
print("SUMMARY: FILTERING IMPACT ON SCAFFOLD FINGERPRINTING")
print("=" * 80)

# Sort by Cohen's d
sorted_metrics = sorted(metrics.items(), key=lambda x: x[1]['cohen_d'], reverse=True)

print(f"\n{'Rank':<5} {'Condition':<30} {'Cohen d':>8} {'SR (%)':>8}")
print("-" * 55)
for i, (label, m) in enumerate(sorted_metrics, 1):
    print(f"{i:<5} {label:<30} {m['cohen_d']:>8.2f} {m['sr_mean']:>8.1f}")

# Check if there's a filtering effect
if len(conditions_1200) >= 2:
    d_vals_1200 = {l: metrics[l]['cohen_d'] for l in conditions_1200}
    best = max(d_vals_1200, key=d_vals_1200.get)
    worst = min(d_vals_1200, key=d_vals_1200.get)
    print(f"\n1200-subject dataset:")
    print(f"  Best filtering:  {best} (d={d_vals_1200[best]:.2f})")
    print(f"  Worst filtering: {worst} (d={d_vals_1200[worst]:.2f})")
    print(f"  Range: {d_vals_1200[best] - d_vals_1200[worst]:.2f}")

print("\n✓ Analysis complete")